# Practice 2 · LAS in Code

**In class** · Goal: run your first code on HaiMa and learn how to interrupt it.

| Marker | Meaning |
|---|---|
| 🟢 EDIT | change this line |
| 🟡 READ | read it, no need to change |
| ⚪ SKIP | run it as it is |


## Question 1 · Pyramid LSTM Downsampling

According to the LAS architecture (Listener), the pBLSTM layer reduces the time resolution of acoustic features to shorten the sequence length while preserving information by concatenating adjacent frames.

Implement the core reshaping logic of `pBLSTMLayer` in the `forward` method using PyTorch.
- `input_x`: Input tensor of shape `(batch_size, timestep, feature_dim)`.
- The goal is to reduce the timestep by half and double the feature dimension.

**Task**: Fill in the missing blocks to compute the new timestep and reshape the tensor.


In [1]:

# Grid Type: Autograded answer
import torch

def pblstm_downsample(input_x):
    batch_size, timestep, feature_dim = input_x.size()
    
    # 🟢 EDIT: Calculate the new timestep (half of the original)
    # YOUR CODE HERE
    new_timestep = timestep // 2
    
    # 🟢 EDIT: Reshape the tensor to (batch_size, new_timestep, feature_dim * 2)
    # YOUR CODE HERE
    output = input_x[:, :new_timestep * 2, :].reshape(
        batch_size, new_timestep, feature_dim * 2
    )
    
    return output
    

In [2]:

# Grid Type: Autograded tests. Points: 10



## Question 2 · Dot-Product Attention Mechanism

In the LAS model, the Attention mechanism computes the context vector based on the decoder state and the listener features. For single-head dot-product attention:
1. Compute the energy (dot-product between decoder state and each listener feature).
2. Apply Softmax to get attention weights.
3. Compute the context vector as the weighted sum of listener features.

**Task**: Fill in the missing blocks to complete the `attention_forward` function.
- `decoder_state`: shape `(batch_size, 1, hidden_dim)`.
- `listener_feature`: shape `(batch_size, T, hidden_dim)`.


In [3]:

# Grid Type: Autograded answer
import torch

def attention_forward(decoder_state, listener_feature):
    # 🟢 EDIT: Compute energy using dot-product (bmm) and squeeze the dimension
    # YOUR CODE HERE
    energy = torch.bmm(
        decoder_state, listener_feature.transpose(1, 2)
    ).squeeze(1)
    
    # 🟢 EDIT: Apply softmax over the time dimension to get attention scores
    # YOUR CODE HERE
    att_score = torch.softmax(energy, dim=-1)
    
    # 🟢 EDIT: Compute the context vector (weighted sum over time)
    # YOUR CODE HERE
    context = torch.bmm(att_score.unsqueeze(1), listener_feature).squeeze(1)
    
    return att_score, context
    

In [4]:

# Grid Type: Autograded tests.



## Question 3 · Speller Teacher Forcing Step

During LAS training, the Speller uses "Teacher Forcing", meaning the ground truth token from the previous step is fed as the input to the current step. 
The input to the RNN at step \(t\) is the concatenation of the one-hot encoded word and the context vector from step \(t\).

**Task**: Fill in the missing blocks to extract the ground truth word and concatenate it with the context vector.
- `ground_truth`: shape `(batch_size, max_label_len, vocab_size)`.
- `context`: shape `(batch_size, hidden_dim)`.
- Return `rnn_input` of shape `(batch_size, 1, vocab_size + hidden_dim)`.


In [5]:

# Grid Type: Autograded answer
import torch

def speller_teacher_forcing_step(ground_truth, context, step):
    # 🟢 EDIT: Get the ground truth word at the current step. Keep the dimension to (batch_size, 1, vocab_size)
    # YOUR CODE HERE
    output_word = ground_truth[:, step:step + 1, :]
    
    # 🟢 EDIT: Concatenate output_word and context (unsqueeze context to dim 1) along the feature dimension
    # YOUR CODE HERE
    rnn_input = torch.cat((output_word, context.unsqueeze(1)), dim=-1)
    
    return rnn_input
    

In [6]:

# Grid Type: Autograded tests.



## Question 4 · Speller Greedy Decoding Step

During inference in the LAS model, the Speller does not have access to the ground truth. Instead, it uses Greedy Decoding: it selects the token with the highest probability at step \(t\) and feeds its one-hot representation as the input to step \(t+1\).

Implement the `greedy_decode_step` function to generate this input.
- `raw_pred`: The predicted probability distribution at step \(t\), shape `(batch_size, vocab_size)`.

**Task**: Fill in the missing blocks to find the max indices, create the one-hot vector, and add the time dimension.


In [7]:

# Grid Type: Autograded answer
import torch

def greedy_decode_step(raw_pred):
    # 🟢 EDIT: Find the index of the maximum probability along the vocabulary dimension (dim=-1)
    # YOUR CODE HERE
    max_indices = torch.argmax(raw_pred, dim=-1)
    
    # 🟢 EDIT: Convert the indices to a one-hot float tensor
    # YOUR CODE HERE
    one_hot = torch.nn.functional.one_hot(
        max_indices, num_classes=raw_pred.size(-1)
    ).float()
    
    # 🟢 EDIT: Add the time step dimension to match the expected RNN input shape (batch_size, 1, vocab_size)
    # YOUR CODE HERE
    rnn_input = one_hot.unsqueeze(1)
    
    return rnn_input
    

In [8]:

# Grid Type: Autograded tests.
